# 6. Descriptive tables and data coverage

**What it does:** joins the cohort key, baseline and outcomes on `row_id` into one file
(`patient_level/analysis_dataset.csv`) and makes the shareable tables:

- **Table 1** by SDH type (overall / traumatic / non-traumatic code)
- **Table 2** by treatment pathway
- **Coverage** of key variables per hospital and per year

Counts under 6 are shown as `<6`, and percentages that would reveal them are blanked.

**Time:** under a minute.

> **Privacy:** everything this notebook *displays* comes from `aggregate/` (counts with cells under 6 suppressed). Files in `patient_level/` stay on GEMINI: don't print their rows, and never paste them outside GEMINI.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# To write the output somewhere else, uncomment and edit BEFORE importing sdh_config:
# os.environ["SDH_OUTPUT_DIR"] = str(Path.home() / "sdh_phenotype_output")

PHENO = Path.cwd().parent              # the phenotype/ folder (this notebook is in phenotype/notebooks/)
sys.path.insert(0, str(PHENO))
import sdh_config as cfg
import sdh_io as io
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 220)
AGG = cfg.AGGREGATE_DIR


def show(name, rows=None, note=None):
    """Display one aggregate (cell-suppressed) table from the output folder."""
    path = AGG / name
    if not path.exists():
        print(f"{name}: not there yet (run the step above first)")
        return None
    df = pd.read_csv(path)
    display(Markdown(f"**{name}**" + (f" - {note}" if note else "")))
    display(df.head(rows) if rows else df)
    return df


def share(names, rows=80):
    """Print aggregate tables as text to paste to Claude / your team (counts only, small cells suppressed)."""
    for name in names:
        path = AGG / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"===== {name} ({len(df)} rows{', first ' + str(rows) if len(df) > rows else ''}) =====")
            print(df.head(rows).to_csv(index=False))
        else:
            print(f"===== {name}: not found =====")

## Run

In [ ]:
%run ../05_tables_and_checks.py

## Results

In [ ]:
show("table1_by_stratum.csv");

**Look for:** plausible distributions (age, sex, mortality, surgery rates) and anything unexpected between traumatic and non-traumatic.

In [ ]:
show("table2_by_pathway.csv");

In [ ]:
show("coverage_by_hospital.csv", note="% of patients with each key variable, per hospital");

In [ ]:
show("coverage_by_year.csv", note="% of patients with each key variable, per year");

**Look for:** hospitals or years where a variable is (nearly) absent - a data gap rather than a clinical difference.

## Share

In [ ]:
share(["table1_by_stratum.csv", "table2_by_pathway.csv", "coverage_by_hospital.csv", "coverage_by_year.csv"], rows=400)